In [1]:
import pandas as pd

df_grid = pd.read_csv(
    r"..\..\Raw\Grid_Stations\2026_04_01_R1-001_Demanda.csv",
    sep=";",
    encoding="utf-8"
)

In [2]:

from pyproj import Transformer

# =========================
# 1. RENOMBRAR COLUMNAS
# =========================
df_grid = df_grid.rename(columns={
    "Coordenada UTM X": "utm_x",
    "Coordenada UTM Y": "utm_y",
    "Capacidad firme disponible (MW)": "capacity_mw"
})

# =========================
# 2. ELIMINAR COLUMNAS INÚTILES
# =========================
df_grid = df_grid.drop(columns=[
    "Posiciones ocupadas",
    "Posiciones libres",
    "Comentarios"
], errors="ignore")

# =========================
# 3. LIMPIAR CAMPOS NUMÉRICOS
# =========================
for col in ["utm_x", "utm_y", "capacity_mw"]:
    df_grid[col] = (
        df_grid[col]
        .astype(str)
        .str.strip()
        .str.replace(".", "", regex=False)   # por si hubiera separador de miles
        .str.replace(",", ".", regex=False)  # coma decimal -> punto
    )
    df_grid[col] = pd.to_numeric(df_grid[col], errors="coerce")

# eliminar filas inválidas en columnas críticas
df_grid = df_grid.dropna(subset=["utm_x", "utm_y", "capacity_mw"])

# capacidad positiva
df_grid = df_grid[df_grid["capacity_mw"] > 0]

# =========================
# 4. CONVERTIR UTM -> LAT/LON
# =========================
transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)

lon, lat = transformer.transform(
    df_grid["utm_x"].values,
    df_grid["utm_y"].values
)

df_grid["longitude"] = lon
df_grid["latitude"] = lat

# =========================
# 5. LIMPIEZA FINAL
# =========================
df_grid = df_grid.drop_duplicates()

# sanity check España
df_grid = df_grid[
    (df_grid["latitude"].between(35, 45)) &
    (df_grid["longitude"].between(-10, 5))
]

df_grid["capacity_kw"] = df_grid["capacity_mw"] * 1000

# =========================
# 6. DATASET FINAL
# =========================
df_grid_clean = df_grid[[
    "latitude",
    "longitude",
    "capacity_mw",
    "capacity_kw"
]].copy()

print("Dimensión:", df_grid_clean.shape)
print(df_grid_clean.head())

Dimensión: (224, 4)
      latitude  longitude  capacity_mw  capacity_kw
153  38.706859  -0.481223        16.42      16420.0
155  38.706859  -0.481223         0.84        840.0
158  38.719256  -0.923730        46.04      46040.0
159  38.681853  -0.493609        18.13      18130.0
161  38.603745  -0.674740        19.25      19250.0
